# Add FamilyOWL 1-hop difficulty columns to existing Excel result files

Put this notebook in:

```text
scripts/analysis/
```

Put the three Excel result files in:

```text
scripts/analysis/xl/
```

Expected files:

```text
abstract_results_FINAL.xlsx
nl_results_FINAL.xlsx
ttl_results_FINAL.xlsx
```

The notebook reads the new difficulty values from:

```text
data/output/FamilyOWL/1hop/SPARQL_questions.csv
```

Then it edits the three Excel files **in place**.  
It does **not** create new Excel files.

It matches rows using:

```text
Task ID
```

Added / overwritten columns:

```text
C1 AxiomTypes
C7 ModalDepth
C8 SignatureDifference
C9 AxiomTypeDiff
Justification Complexity Score
```

In [1]:
from pathlib import Path
import csv
from copy import copy
from typing import Any

try:
    from openpyxl import load_workbook
except ImportError as e:
    raise ImportError(
        "openpyxl is required. Install it with: pip install openpyxl"
    ) from e

In [2]:
# ============================================================
# Configuration
# ============================================================

DIFFICULTY_COLUMNS = [
    "C1 AxiomTypes",
    "C7 ModalDepth",
    "C8 SignatureDifference",
    "C9 AxiomTypeDiff",
    "Justification Complexity Score",
]

EXCEL_FILENAMES = [
    "abstract_results_FINAL.xlsx",
    "nl_results_FINAL.xlsx",
    "ttl_results_FINAL.xlsx",
]

SPARQL_RELATIVE_PATH = Path("data") / "output" / "FamilyOWL" / "1hop" / "SPARQL_questions.csv"


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for parent in [current] + list(current.parents):
        if (parent / "pom.xml").exists() and (parent / "data").exists():
            return parent
    raise FileNotFoundError(
        "Could not find repo root. Make sure this notebook is inside the CORE-LLM-Bench-MasterProject2 repo."
    )


# This works whether VS Code/Jupyter starts in scripts/analysis or in the repo root.
REPO_ROOT = find_repo_root(Path.cwd())
ANALYSIS_DIR = REPO_ROOT / "scripts" / "analysis"
XL_DIR = ANALYSIS_DIR / "xl"
SPARQL_CSV = REPO_ROOT / SPARQL_RELATIVE_PATH

EXCEL_PATHS = [XL_DIR / name for name in EXCEL_FILENAMES]

print("Repo root:", REPO_ROOT)
print("SPARQL CSV:", SPARQL_CSV)
print("Excel folder:", XL_DIR)
print()
print("Excel files:")
for path in EXCEL_PATHS:
    print(" -", path)

Repo root: C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2
SPARQL CSV: C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\data\output\FamilyOWL\1hop\SPARQL_questions.csv
Excel folder: C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl

Excel files:
 - C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl\abstract_results_FINAL.xlsx
 - C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl\nl_results_FINAL.xlsx
 - C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl\ttl_results_FINAL.xlsx


In [3]:
# ============================================================
# Small helpers
# ============================================================

def clean_header(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def header_key(value: Any) -> str:
    return clean_header(value).lower()


def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")


def parse_cell_value(value: Any) -> Any:
    """Convert numeric-looking CSV values to int where possible."""
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    try:
        as_float = float(text)
        if as_float.is_integer():
            return int(as_float)
        return as_float
    except ValueError:
        return text


def copy_cell_style(source_cell, target_cell) -> None:
    """Copy style from one cell to another without copying the value."""
    if source_cell is None or target_cell is None:
        return

    if source_cell.has_style:
        target_cell.font = copy(source_cell.font)
        target_cell.fill = copy(source_cell.fill)
        target_cell.border = copy(source_cell.border)
        target_cell.alignment = copy(source_cell.alignment)
        target_cell.number_format = source_cell.number_format
        target_cell.protection = copy(source_cell.protection)

In [4]:
# ============================================================
# Load difficulty values from SPARQL_questions.csv
# ============================================================

require_file(SPARQL_CSV)

with SPARQL_CSV.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    raw_fieldnames = reader.fieldnames or []

    # Map stripped column names to actual CSV field names.
    field_map = {clean_header(name): name for name in raw_fieldnames}

    if "Task ID" not in field_map:
        raise ValueError(f"SPARQL CSV does not contain a Task ID column. Columns found: {raw_fieldnames}")

    missing = [col for col in DIFFICULTY_COLUMNS if col not in field_map]
    if missing:
        raise ValueError(
            "SPARQL CSV is missing difficulty columns: "
            + ", ".join(missing)
            + "\nColumns found: "
            + ", ".join(raw_fieldnames)
        )

    task_id_field = field_map["Task ID"]
    difficulty_field_map = {col: field_map[col] for col in DIFFICULTY_COLUMNS}

    difficulty_by_task_id = {}
    duplicate_task_ids = set()

    for row in reader:
        task_id = clean_header(row.get(task_id_field))

        if not task_id:
            continue

        if task_id in difficulty_by_task_id:
            duplicate_task_ids.add(task_id)

        difficulty_by_task_id[task_id] = {
            col: parse_cell_value(row.get(actual_field))
            for col, actual_field in difficulty_field_map.items()
        }

print("Difficulty rows loaded:", len(difficulty_by_task_id))
print("Duplicate Task IDs in SPARQL CSV:", len(duplicate_task_ids))

if duplicate_task_ids:
    print("Example duplicates:", list(sorted(duplicate_task_ids))[:5])

first_key = next(iter(difficulty_by_task_id))
print("\nExample Task ID:", first_key)
print("Example values:", difficulty_by_task_id[first_key])

Difficulty rows loaded: 15133
Duplicate Task IDs in SPARQL CSV: 7906
Example duplicates: ['1hop-Person_john_william_folland_john_william_folland-john_william_folland-isMalePartnerIn-BIN', '1hop-Person_john_william_folland_john_william_folland-john_william_folland-isMalePartnerIn-MC', '1hop-Person_john_william_folland_john_william_folland-john_william_folland-isPartnerIn-BIN', '1hop-Person_john_william_folland_john_william_folland-john_william_folland-isPartnerIn-MC', '1hop-Person_john_william_folland_john_william_folland-john_william_folland-rdf:type-BIN']

Example Task ID: 1hop-Person_john_william_folland_john_william_folland-john_william_folland-isBloodrelationOf-BIN
Example values: {'C1 AxiomTypes': 2, 'C7 ModalDepth': 0, 'C8 SignatureDifference': 0, 'C9 AxiomTypeDiff': 0, 'Justification Complexity Score': 200}


In [5]:
# ============================================================
# Update one Excel workbook in place
# ============================================================

def find_header_columns(ws, header_row: int = 1) -> dict[str, int]:
    """Return mapping from normalized header text to column index."""
    mapping = {}

    for col_idx in range(1, ws.max_column + 1):
        value = ws.cell(row=header_row, column=col_idx).value
        key = header_key(value)

        if key:
            mapping[key] = col_idx

    return mapping


def update_workbook_in_place(path: Path) -> dict[str, Any]:
    require_file(path)

    wb = load_workbook(path)

    # Find the first worksheet that has a Task ID header in row 1.
    target_ws = None
    target_headers = None

    for ws in wb.worksheets:
        headers = find_header_columns(ws, header_row=1)
        if "task id" in headers:
            target_ws = ws
            target_headers = headers
            break

    if target_ws is None:
        raise ValueError(f"No worksheet with a 'Task ID' header found in {path}")

    ws = target_ws
    headers = target_headers
    task_id_col = headers["task id"]

    # Add missing difficulty headers at the end, or reuse existing columns.
    difficulty_col_indices = {}

    for col_name in DIFFICULTY_COLUMNS:
        key = header_key(col_name)

        if key in headers:
            difficulty_col_indices[col_name] = headers[key]
        else:
            new_col = ws.max_column + 1
            difficulty_col_indices[col_name] = new_col

            # Add header.
            header_cell = ws.cell(row=1, column=new_col)
            header_cell.value = col_name

            # Copy header style from previous column if possible.
            source_header_cell = ws.cell(row=1, column=new_col - 1)
            copy_cell_style(source_header_cell, header_cell)

            # Give a sensible width.
            previous_letter = source_header_cell.column_letter
            new_letter = header_cell.column_letter
            previous_width = ws.column_dimensions[previous_letter].width
            ws.column_dimensions[new_letter].width = max(previous_width or 12, min(len(col_name) + 2, 32))

            # Update local header mapping.
            headers[key] = new_col

    matched_rows = 0
    unmatched_rows = 0
    empty_task_id_rows = 0

    # Write difficulty values row by row.
    for row_idx in range(2, ws.max_row + 1):
        task_id = clean_header(ws.cell(row=row_idx, column=task_id_col).value)

        if not task_id:
            empty_task_id_rows += 1
            continue

        values = difficulty_by_task_id.get(task_id)

        if values is None:
            unmatched_rows += 1
            continue

        matched_rows += 1

        for col_name, col_idx in difficulty_col_indices.items():
            target_cell = ws.cell(row=row_idx, column=col_idx)
            target_cell.value = values[col_name]

            # If this was a newly-added column or an empty-style cell, copy style from the previous column.
            source_cell = ws.cell(row=row_idx, column=max(1, col_idx - 1))
            copy_cell_style(source_cell, target_cell)

    wb.save(path)

    return {
        "file": path.name,
        "worksheet": ws.title,
        "rows_total": max(ws.max_row - 1, 0),
        "matched_rows": matched_rows,
        "unmatched_rows": unmatched_rows,
        "empty_task_id_rows": empty_task_id_rows,
        "difficulty_columns": difficulty_col_indices,
    }

In [6]:
# ============================================================
# Edit the three Excel files IN PLACE
# ============================================================

for path in EXCEL_PATHS:
    require_file(path)

reports = []

for path in EXCEL_PATHS:
    print(f"Updating: {path}")
    report = update_workbook_in_place(path)
    reports.append(report)
    print("  worksheet:", report["worksheet"])
    print("  matched rows:", report["matched_rows"])
    print("  unmatched rows:", report["unmatched_rows"])
    print("  empty Task ID rows:", report["empty_task_id_rows"])
    print()

print("Done. The existing Excel files were edited in place.")

Updating: C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl\abstract_results_FINAL.xlsx
  worksheet: abstract_results_FINAL.csv
  matched rows: 1401
  unmatched rows: 23
  empty Task ID rows: 0

Updating: C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl\nl_results_FINAL.xlsx
  worksheet: nl_results_FINAL.csv
  matched rows: 1376
  unmatched rows: 23
  empty Task ID rows: 0

Updating: C:\Users\abcma\OneDrive\Desktop\pro2\CORE-LLM-Bench-MasterProject2\scripts\analysis\xl\ttl_results_FINAL.xlsx
  worksheet: ttl_results_FINAL.csv
  matched rows: 1401
  unmatched rows: 23
  empty Task ID rows: 0

Done. The existing Excel files were edited in place.


In [7]:
# ============================================================
# Verification: show the first few updated values from each file
# ============================================================

for path in EXCEL_PATHS:
    wb = load_workbook(path, data_only=False)

    ws = None
    headers = None

    for candidate in wb.worksheets:
        candidate_headers = find_header_columns(candidate, header_row=1)
        if "task id" in candidate_headers:
            ws = candidate
            headers = candidate_headers
            break

    if ws is None:
        print(f"{path.name}: no sheet with Task ID found")
        continue

    task_id_col = headers["task id"]
    difficulty_cols = {
        col_name: headers[header_key(col_name)]
        for col_name in DIFFICULTY_COLUMNS
        if header_key(col_name) in headers
    }

    print("=" * 100)
    print(path.name, "| sheet:", ws.title)
    print("Columns found:", list(difficulty_cols.keys()))
    print("-" * 100)

    shown = 0
    for row_idx in range(2, ws.max_row + 1):
        task_id = clean_header(ws.cell(row=row_idx, column=task_id_col).value)
        if not task_id:
            continue

        values = {
            col_name: ws.cell(row=row_idx, column=col_idx).value
            for col_name, col_idx in difficulty_cols.items()
        }

        print(task_id, "=>", values)

        shown += 1
        if shown >= 5:
            break

    print()

abstract_results_FINAL.xlsx | sheet: abstract_results_FINAL.csv
Columns found: ['C1 AxiomTypes', 'C7 ModalDepth', 'C8 SignatureDifference', 'C9 AxiomTypeDiff', 'Justification Complexity Score']
----------------------------------------------------------------------------------------------------
1hop-Thing_ada_rachel_heath_1868_gwendoline_heath_1878-gwendoline_heath_1878-rdf:type-BIN => {'C1 AxiomTypes': 4, 'C7 ModalDepth': 0, 'C8 SignatureDifference': 0, 'C9 AxiomTypeDiff': 1, 'Justification Complexity Score': 450}
1hop-Thing_ada_rachel_heath_1868_gwendoline_heath_1878-gwendoline_heath_1878-rdf:type-MC => {'C1 AxiomTypes': 4, 'C7 ModalDepth': 0, 'C8 SignatureDifference': 0, 'C9 AxiomTypeDiff': 1, 'Justification Complexity Score': 450}
1hop-Thing_ada_rachel_heath_1868_gwendoline_heath_1878-gwendoline_heath_1878-rdf:type-BIN => {'C1 AxiomTypes': 4, 'C7 ModalDepth': 0, 'C8 SignatureDifference': 0, 'C9 AxiomTypeDiff': 1, 'Justification Complexity Score': 450}
1hop-Thing_ada_rachel_heath_186

## Notes

- The notebook edits the Excel files directly.
- It matches by `Task ID`.
- It does not run the LLM again.
- It does not create new Excel files.
- If a row in Excel has a `Task ID` that does not exist in `SPARQL_questions.csv`, that row is left unchanged for the new difficulty columns.
- Re-running the notebook is safe: it overwrites the same five columns instead of duplicating them.